# House Prices — Step 4: Target Encoding

Target Encoding replaces each category with a statistic of the target variable (`SalePrice`) — typically the **mean** — computed from the rows belonging to that category.

It's especially useful for **high-cardinality nominal columns** like `Neighborhood` (25 categories), where One-Hot Encoding would add dozens of columns: Target Encoding compresses the whole column into a single, target-informative number instead.

**Important risk:** if a category's mean is computed using *all* rows — including the row being encoded — the target leaks into the feature for that row, which can cause overfitting, especially for rare categories with few observations. This notebook uses **smoothing** to reduce that risk by blending each category's mean with the global mean. A full fix — **K-Fold Target Encoding**, which removes the leakage entirely — is covered in the next notebook.

## 1. Load the cleaned dataset

We need `SalePrice` (the target) alongside the categorical columns, so we continue from `train_cleaned.csv` (Step 1).

In [1]:
import pandas as pd
pd.set_option('display.max_columns', 15)

df = pd.read_csv('train_cleaned.csv')
print("Shape:", df.shape)
df[['Neighborhood', 'SalePrice']].head(5)

Shape: (1460, 81)


,Neighborhood,SalePrice
0,CollgCr,208500
1,Veenker,181500
2,CollgCr,223500
3,Crawfor,140000
4,NoRidge,250000


## 2. Basic idea: mean `SalePrice` per category

Let's start simple: for `Neighborhood`, compute the average `SalePrice` within each neighborhood. Notice how this single number already captures a lot of signal — e.g. `NridgHt` and `NoRidge` (expensive areas) get a high value, while cheaper neighborhoods get a low value.

In [2]:
neighborhood_means = df.groupby('Neighborhood')['SalePrice'].mean().sort_values(ascending=False)
neighborhood_means.head(10)

Neighborhood
NoRidge    335295.317073
NridgHt    316270.623377
StoneBr    310499.000000
Timber     242247.447368
Veenker    238772.727273
Somerst    225379.837209
ClearCr    212565.428571
Crawfor    210624.725490
CollgCr    197965.773333
Blmngtn    194870.882353
Name: SalePrice, dtype: float64

## 3. Why this can be risky: rare categories

Some categories have very few rows. Their "mean" is based on so little data that it can be noisy or even just memorize the target for those specific rows (an extreme overfitting risk). Let's check how many rows back each neighborhood's mean.

In [3]:
counts = df['Neighborhood'].value_counts().sort_values()
print("Neighborhoods with the fewest observations:")
counts.head(5)

Neighborhoods with the fewest observations:


Neighborhood
Blueste     2
NPkVill     9
Veenker    11
BrDale     16
MeadowV    17
Name: count, dtype: int64

## 4. Smoothed target encoding

To reduce the influence of small categories, we blend each category's mean with the **global mean**, weighted by how many observations that category has:

`smoothed_mean = (count * category_mean + m * global_mean) / (count + m)`

Here `m` is a smoothing parameter: larger `m` pulls small categories more strongly toward the global average. We'll use `m = 10` as a reasonable default.

In [4]:
def smoothed_target_encode(series, target, m=10):
    global_mean = target.mean()
    agg = target.groupby(series).agg(['mean', 'count'])
    smoothed = (agg['count'] * agg['mean'] + m * global_mean) / (agg['count'] + m)
    return series.map(smoothed), smoothed

encoded_neighborhood, mapping = smoothed_target_encode(df['Neighborhood'], df['SalePrice'], m=10)

comparison = pd.DataFrame({
    'Neighborhood': df['Neighborhood'],
    'Raw mean': df['Neighborhood'].map(df.groupby('Neighborhood')['SalePrice'].mean()),
    'Smoothed mean': encoded_neighborhood
})
comparison.drop_duplicates().sort_values('Smoothed mean', ascending=False).head(10)

,Neighborhood,Raw mean,Smoothed mean
4,NoRidge,335295.317073,305025.881547
11,NridgHt,316270.623377,300713.217918
58,StoneBr,310499.000000,273476.770254
41,Timber,242247.447368,229471.144977
6,Somerst,225379.837209,220748.728739
1,Veenker,238772.727273,211224.378995
3,Crawfor,210624.725490,205755.294408
69,ClearCr,212565.428571,204237.998919
0,CollgCr,197965.773333,196900.487243
50,Gilbert,192854.506329,191513.684932


## 5. Apply smoothed Target Encoding to all categorical columns

We now apply the same smoothed target encoding to every categorical column, producing a fully numeric dataframe where each original text column becomes a single numeric column (unlike One-Hot Encoding, this doesn't add extra columns).

In [5]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Encoding {len(cat_cols)} categorical columns")

df_target_encoded = df.copy()
target_mappings = {}

for col in cat_cols:
    encoded_col, mapping = smoothed_target_encode(df[col], df['SalePrice'], m=10)
    df_target_encoded[col] = encoded_col
    target_mappings[col] = mapping

print("Shape unchanged:", df_target_encoded.shape)
df_target_encoded.head(3)

Encoding 43 categorical columns
Shape unchanged: (1460, 81)


/tmp/ipykernel_96/1698950818.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,...,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,190918.140361,65.0,8450,181129.108578,NaN,...,NaN,0,2,2008,173460.719623,175249.562052,208500
1,2,20,190918.140361,80.0,9600,181129.108578,NaN,...,NaN,0,5,2007,173460.719623,175249.562052,181500
2,3,60,190918.140361,68.0,11250,181129.108578,NaN,...,NaN,0,9,2008,173460.719623,175249.562052,223500


## 6. Save the target-encoded dataset

Saved separately so we can compare against the Label Encoding, One-Hot Encoding, and (next) K-Fold Target Encoding versions.

Note: this version's encodings were computed using the **full training set**, including each row's own target — that's the leakage risk described above. The next notebook builds the K-Fold version, which is the version you'd actually want to use for model training.

In [6]:
df_target_encoded.to_csv('train_target_encoded.csv', index=False)
print("Saved train_target_encoded.csv with shape:", df_target_encoded.shape)

Saved train_target_encoded.csv with shape: (1460, 81)
